# Sliding Window Attention — Exercise

Companion to [Attention Part 3 § Sliding Window Attention](https://tanulsingh.github.io/blog/attention-sparsity#sliding-window-attention-mistral-2023).

Each token attends to only its `w` nearest neighbors in the past (causal window). Compute drops from `O(n²)` to `O(n·w)` — linear when `w` is fixed.

Used by Mistral 7B (window=4096) and many other models.

The trick: build a mask that's `True` where `j <= i AND i - j < w`, and apply it to standard attention.

## Setup

In [ ]:
# TODO: imports — torch, torch.nn as nn, torch.nn.functional as F, math

## TODO 1 — `sliding_window_mask`

Build a `(seq_len, seq_len)` bool mask:
- `mask[i, j] == True` iff `j <= i` (causal) AND `i - j < window_size` (within window)

For `seq_len=6, window_size=3`, the mask should be:
```
[[T, F, F, F, F, F],
 [T, T, F, F, F, F],
 [T, T, T, F, F, F],
 [F, T, T, T, F, F],   ← position 3 can't see position 0 (distance 3 >= window 3)
 [F, F, T, T, T, F],
 [F, F, F, T, T, T]]
```

In [ ]:
def sliding_window_mask(seq_len: int, window_size: int) -> 'torch.Tensor':
    """Return shape (seq_len, seq_len) bool mask."""
    # TODO 1: implement
    # Hint: for two index vectors i, j, build a 2D matrix of (i - j) and check
    #       (i - j >= 0) AND (i - j < window_size)
    pass

In [ ]:
# Sanity check:
#   - sliding_window_mask(6, 3) matches the example above
#   - Diagonal is True (token sees itself)
#   - Position 5 sees positions [3, 4, 5] only

## TODO 2 — `SlidingWindowAttention` module

A Multi-Head Attention variant that uses the sliding window mask. Just plug your mask into the standard attention computation.

In [ ]:
class SlidingWindowAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int, window_size: int):
        super().__init__()
        # TODO 2a: store dimensions and window_size
        # TODO 2b: W_q, W_k, W_v, W_o as nn.Linear(d_model, d_model)
        pass

    def forward(self, x):
        # x: (batch, seq, d_model)
        # TODO 2c: project to Q, K, V and reshape to (batch, n_heads, seq, d_head)
        # TODO 2d: build sliding_window_mask(seq, window_size)
        # TODO 2e: standard scaled dot-product attention with the mask
        # TODO 2f: reshape back, apply W_o
        pass

In [ ]:
# Sanity check:
#   - swa = SlidingWindowAttention(d_model=64, n_heads=8, window_size=4)
#   - out = swa(torch.randn(2, 16, 64))
#   - out.shape == (2, 16, 64)
#   - When seq_len <= window_size, this should match plain causal attention
#   - When seq_len > window_size, distant tokens are completely blocked

## TODO 3 — Verify the locality property

For `window_size=3`, position 5 should only depend on values at positions [3, 4, 5]. Verify this by changing `V[2]` and confirming `output[5]` is unchanged.

In [ ]:
# TODO 3:
#   - Create x_a of shape (1, 8, 64), then x_b which differs from x_a only at position 2
#   - Forward both through SlidingWindowAttention(window_size=3)
#   - Assert out_a[0, 5] ≈ out_b[0, 5]   (position 5 doesn't see position 2)
#   - Assert out_a[0, 4] ≈ out_b[0, 4]   (position 4 doesn't see position 2; 4-2 == window_size)
#   - Assert out_a[0, 3] ≠ out_b[0, 3]   (position 3 DOES see position 2)

## Run the tests

In [ ]:
from tests import run_sliding_window_tests
# run_sliding_window_tests(sliding_window_mask, SlidingWindowAttention)